# Limpieza de datos – Top 50 goleadores del Mundial 2026

**Proyecto:** Adivina al Goleador (equipo Balón de Oro) · Algoritmia y Programación 2026-2

## Fuente y temática
- **Archivo original:** `data/raw/base_datos_original.xlsx` (hoja *Goleadores*), con los 50 máximos goleadores del Mundial 2026.
- **Fuentes de la información:** goles y partidos de losmundialesdefutbol.com; asistencias y faltas de futboltotal.futbol (API-Sports); fecha de nacimiento y posición de Wikipedia (plantillas del Mundial 2026).
- **Temática:** deportes (fútbol).

## Objetivo
Dejar en `data/datos.xlsx` (Excel) únicamente seis variables: **edad, posición, goles, partidos jugados, asistencias y faltas**.

## Variables que se conservan
| Variable final | Significado | Tipo |
|---|---|---|
| `edad` | Edad del jugador (años cumplidos al 6-nov-2026) | entero |
| `posicion` | Delantero, Mediocampista o Defensa | texto |
| `goles` | Goles marcados en el Mundial 2026 | entero |
| `partidos` | Partidos jugados en el Mundial | entero |
| `asistencias` | Asistencias en el Mundial | entero (puede estar vacío) |
| `faltas` | Faltas cometidas en el Mundial | entero (puede estar vacío) |


1. Importar librerias y ruta


In [5]:
from pathlib import Path 

import pandas as pd 
from openpyxl.styles import Alignment, Font, PatternFill 

DATA = Path("data") if (Path("data") / "raw").exists() else Path(".")
RUTA_EXCEL = DATA / "raw" / "base_datos_original.xlsx"
RUTA_XLSX_LIMPIO = DATA / "datos.xlsx"
RUTA_CSV_LIMPIO = DATA / "datos.csv"
print("leyendo:", RUTA_EXCEL) 

leyendo: raw\base_datos_original.xlsx


2. CARGAR EL EXCEL 
Las dos primeras filas de la hoja son un título y la fecha de referencia de la edad; los encabezados reales están en la **fila 3**, por eso se usa `header=2`.

In [6]:
try:
    df = pd.read_excel(RUTA_EXCEL, header=2)
except FileNotFoundError:
    raise SystemExit(f"no se encontro {RUTA_EXCEL}. Copia el excel en data/raw/. ")
print(df.shape)
df.head()

(60, 20)


,Pos.,Jugador,Selección,Posición,Fecha nac.,Edad,Club actual (nov-2026),Goles Mundial,Asist.,G+A,Partidos,Goles/PJ,Faltas cometidas,Faltas/PJ,Amarillas,Rojas,Goles selección (pre-Mundial),Goles selección (con Mundial),Club antes del Mundial (jun-2026),Verificación del club
0,1,Kylian Mbappé,Francia,DEL,1998-12-20,27.0,Real Madrid,10.0,4,14,8.0,1.250,6,0.75,1,0,56.0,66.0,Real Madrid,Sin cambio detectado
1,2,Lionel Messi,Argentina,DEL,1987-06-24,39.0,Inter Miami CF,8.0,4,12,8.0,1.000,3,0.375,0,0,117.0,125.0,Inter Miami CF,Sin cambio detectado
2,3,Jude Bellingham,Inglaterra,MED,2003-06-29,23.0,Real Madrid,7.0,1,8,8.0,0.875,9,1.125,1,0,6.0,13.0,Real Madrid,Sin cambio detectado
3,3,Erling Haaland,Noruega,DEL,2000-07-21,26.0,Manchester City,7.0,0,7,5.0,1.400,5,1,0,0,55.0,62.0,Manchester City,Sin cambio detectado
4,5,Ousmane Dembélé,Francia,DEL,1997-05-15,29.0,Paris Saint-Germain,6.0,2,8,8.0,0.750,0,0,0,0,7.0,13.0,Paris Saint-Germain,Sin cambio detectado


3. EXPLORACION INICIAL
Se revisan tipos de datos, valores vacíos y filas que no son jugadores (totales y notas al pie).

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 20 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   Pos.                               58 non-null     object        
 1   Jugador                            50 non-null     str           
 2   Selección                          51 non-null     str           
 3   Posición                           50 non-null     str           
 4   Fecha nac.                         50 non-null     datetime64[us]
 5   Edad                               50 non-null     float64       
 6   Club actual (nov-2026)             50 non-null     str           
 7   Goles Mundial                      51 non-null     float64       
 8   Asist.                             51 non-null     object        
 9   G+A                                51 non-null     object        
 10  Partidos                           50 non-null     

In [8]:
print("valores vacios por columna: ")
print(df.isna().sum())
print()
print("ultimas filas (totales y notas):")
df.tail(8)[["Pos.", "Jugador", "Selección", "Goles Mundial"]]

valores vacios por columna: 
Pos.                                  2
Jugador                              10
Selección                             9
Posición                             10
Fecha nac.                           10
Edad                                 10
Club actual (nov-2026)               10
Goles Mundial                         9
Asist.                                9
G+A                                   9
Partidos                             10
Goles/PJ                             10
Faltas cometidas                      9
Faltas/PJ                            10
Amarillas                             9
Rojas                                 9
Goles selección (pre-Mundial)        10
Goles selección (con Mundial)         9
Club antes del Mundial (jun-2026)    10
Verificación del club                10
dtype: int64

ultimas filas (totales y notas):


,Pos.,Jugador,Selección,Goles Mundial
52,Notas:,NaN,NaN,NaN
53,Fuentes: goles y partidos de losmundialesdefut...,NaN,NaN,NaN
54,Posición según la lista oficial FIFA: DEL = de...,NaN,NaN,NaN
55,Club actual: 5 traspasos confirmados en el mer...,NaN,NaN,NaN
56,Goles de selección = goles internacionales de ...,NaN,NaN,NaN
57,s/d (celdas amarillas) = sin dato fiable de as...,NaN,NaN,NaN
58,Mostafa Zico: se asumió Mostafa Ziko (Pyramids...,NaN,NaN,NaN
59,Puestos 29–50: jugadores con 2 goles (22 de 29...,NaN,NaN,NaN


In [9]:
# celdas con s/d ( sin datos) en las columnas numericas que nos interesan 
for col in ["Asist.", "Faltas cometidas"]:
    sd = df[df[col].astype(str).str.strip().str.lower() == "s/d"]
    print(col, "->", sd["Jugador"].tolist())

Asist. -> ['Lautaro Martínez']
Faltas cometidas -> ['Lautaro Martínez']


4. QUITAR LAS FILAS QUE NO SON JUGADORES
Al final de la hoja hay una fila de **Totales** y varias **notas**. Solo se conservan las filas que tienen nombre de jugador.
Antes de borrar la fila de totales se guarda, porque más adelante sirve para **verificar** que la suma de los datos coincide.

In [10]:
filas_totales = df[df["Selección"] == "Totales"].iloc[0]

df = df[df["Jugador"].notna()].copy()
print("Jugadores:", len(df))

Jugadores: 50


5. LIMPIEZA
**Decisiones tomadas:**
**Nombres de columnas:** se renombran en minúscula y sin tildes ni espacios, para escribirlos fácil en el código de la app.
 **`s/d` (sin dato):** se convierte en valor vacío (`NaN`). Se hace así porque la fuente no tiene un dato fiable para Lautaro Martínez en asistencias y faltas. **No se inventa ni se rellena con ceros**, porque un 0 diría algo falso.
 **Posición:** las siglas DEL, MED y DEF se cambian por nombres completos para que se entiendan en la app.
 **Tipos de datos:** las variables numéricas pasan a enteros; las de asistencias y faltas usan `Int64` para admitir el valor vacío sin convertirse en decimales.

In [11]:
df = df.rename(columns={
    "Jugador": "jugador",
    "Posición": "posicion",
    "Fecha nac.": "fecha_nac",
    "Edad": "edad",
    "Goles Mundial": "goles",
    "Asist.": "asistencias",
    "Partidos": "partidos",
    "Faltas cometidas": "faltas",

})
# 's/d' -> vacio (NaN) y conversion a numero
for col in ["asistencias", "faltas"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# siglas -> nombres completos
df["posicion"] = df["posicion"].map({
    "DEL": "Delantero",
    "MED": "Mediocampista",
    "DEF": "Defensa",
})

# tipos enteros
for col in ["edad", "goles", "partidos"]:
    df[col] = df[col].astype(int)

df["fecha_nac"] = pd.to_datetime(df["fecha_nac"])
df[["jugador", "posicion", "edad", "goles", "partidos", "asistencias", "faltas"]].dtypes

jugador          str
posicion         str
edad           int64
goles          int64
partidos       int64
asistencias    Int64
faltas         Int64
dtype: object

5. VERIFICACION DE LA CALIDAD
Se comprueba que los datos sean coherentes antes de exportar.

In [12]:
# no hay jugadores repetidos
print("Jugadores repetidos:", df["jugador"].duplicated().sum())

# todas las posiciones se reconocieron
print("Posiciones sin reconocer:", df["posicion"].isna().sum())

#La edad coincide con la calculada desde la fecha de nacimiento (referencia: 6-nov-2026)
referencia = pd.Timestamp("2026-11-06")
edad_calculada = (
    referencia.year - df["fecha_nac"].dt.year
    - ((df["fecha_nac"].dt.month > referencia.month) |
       ((df["fecha_nac"].dt.month == referencia.month) &  (df["fecha_nac"].dt.day > referencia.day)))
)
print("Edades que no coinciden:", (edad_calculada != df["edad"]).sum())

# las sumas coinciden con las fila de totales del excel
print("Goles:", df["goles"].sum(), "| Excel:", filas_totales["Goles Mundial"])
print("Asistencias:", df["asistencias"].sum(), "| Excel:", filas_totales["Asist."])
print("Faltas:", df["faltas"].sum(), "| Excel:", filas_totales["Faltas cometidas"])

Jugadores repetidos: 0
Posiciones sin reconocer: 0
Edades que no coinciden: 0
Goles: 159 | Excel: 159.0
Asistencias: 39 | Excel: 39
Faltas: 184 | Excel: 184


6. SELECCIONAR LAS COLUMNAS FINALES Y EXPORTAR EXCEL
Solo se dejan las seis variables pedidas. La columna `jugador` se usó para verificar la calidad, pero no pasa al archivo final.

El archivo se guarda como **Excel (`data/datos.xlsx`)**, con el encabezado resaltado en el verde de la aplicación y la primera fila fija.

In [13]:
COLUMNAS_FINALES = ["edad", "posicion", "goles", "partidos", "asistencias", "faltas"]
GUARDAR_TAMBIEN_CSV = True

datos = df[COLUMNAS_FINALES]

with pd.ExcelWriter(RUTA_XLSX_LIMPIO, engine="openpyxl") as escritor:
    datos.to_excel(escritor, sheet_name="Datos", index=False)
    hoja = escritor.sheets["Datos"]
    hoja.freeze_panes = "A2"
    for celda in hoja[1]:
        celda.font = Font(bold=True, color="FFFFFF")
        celda.fill = PatternFill("solid", fgColor="0B5D3B")
        celda.alignment = Alignment(horizontal="center")
    for columna in hoja.columns:
        hoja.column_dimensions[columna[0].column_letter].width = 16

if GUARDAR_TAMBIEN_CSV:
    datos.to_csv(RUTA_CSV_LIMPIO, index=False, encoding="utf-8")
print("Archivo final:", RUTA_CSV_LIMPIO, "-", datos.shape[0], "filas y", datos.shape[1], "columnas")

Archivo final: datos.csv - 50 filas y 6 columnas


In [14]:
# comprobar que el excel se puede volver a leer correctamente
comprobacion = pd.read_excel(RUTA_XLSX_LIMPIO)
print(comprobacion.shape)
comprobacion.head(10)

(50, 6)


,edad,posicion,goles,partidos,asistencias,faltas
0,27,Delantero,10,8,4.0,6.0
1,39,Delantero,8,8,4.0,3.0
2,23,Mediocampista,7,8,1.0,9.0
3,26,Delantero,7,5,0.0,5.0
4,29,Delantero,6,8,2.0,0.0
5,33,Delantero,6,7,1.0,8.0
6,29,Delantero,5,8,1.0,4.0
7,29,Delantero,4,5,1.0,5.0
8,28,Delantero,4,4,1.0,1.0
9,26,Delantero,4,5,1.0,4.0


In [15]:
# vacios que quedan en el archivo final (esperados: 1 en asistencias y 1 en faltas)
comprobacion.isna().sum()

edad           0
posicion       0
goles          0
partidos       0
asistencias    1
faltas         1
dtype: int64

Resultado
- `data/datos.xlsx`: base limpia en Excel con **edad, posición, goles, partidos, asistencias y faltas** de 50 jugadores.
- El único dato vacío corresponde a asistencias y faltas de Lautaro Martínez (sin dato fiable en la fuente)